# Experiment: Query -> Dense/BM25 -> Graph Expansion -> PCST

This notebook tests the retrieval concept end-to-end on the EU AI Act graph artifacts.

Pipeline:
1. Encode query and run dense + BM25 retrieval.
2. Fuse the two rankings into seed nodes.
3. Expand a k-hop subgraph around seeds.
4. Apply PCST pruning to keep a compact, relevant subgraph.



In [ ]:
# Setup: imports and project path (self-contained, no src/retrieval dependency)
from pathlib import Path
import re
import hashlib

import numpy as np
import pandas as pd
import networkx as nx
import yaml
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity

try:
    from rank_bm25 import BM25Okapi
except ImportError as exc:
    raise ImportError('rank_bm25 is required. Install with: pip install rank-bm25') from exc

try:
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError('sentence-transformers is required. Install with: pip install sentence-transformers') from exc

try:
    from pcst_fast import pcst_fast
    PCST_AVAILABLE = True
except ImportError:
    pcst_fast = None
    PCST_AVAILABLE = False

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'configs').exists() and (REPO_ROOT / '..' / 'configs').exists():
    REPO_ROOT = (REPO_ROOT / '..').resolve()


def resolve_path(path_str: str) -> Path:
    p = Path(path_str)
    return p if p.is_absolute() else (REPO_ROOT / p)


def _tokenize(text: str) -> list[str]:
    return re.findall(r"\b\w+\b", str(text).lower())


def _normalize(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr[None, :]
    n = np.linalg.norm(arr, axis=1, keepdims=True)
    n = np.maximum(n, 1e-12)
    return arr / n


class LocalDenseRetriever:
    def __init__(self, nodes_df: pd.DataFrame, embeddings: np.ndarray, encoder: SentenceTransformer):
        self.nodes_df = nodes_df.reset_index(drop=True).copy()
        self.node_ids = self.nodes_df['node_id'].astype(str).tolist()
        self.embeddings = _normalize(np.asarray(embeddings, dtype=np.float32))
        self.encoder = encoder
        self.id_to_text = dict(zip(self.node_ids, self.nodes_df.get('text', pd.Series([''] * len(self.nodes_df))).fillna('').astype(str)))
        self.id_to_type = dict(zip(self.node_ids, self.nodes_df.get('type', pd.Series([''] * len(self.nodes_df))).fillna('').astype(str)))
        self._query_cache = {}

    def encode_query(self, query: str) -> np.ndarray:
        q = str(query).strip()
        key = hashlib.sha1(q.encode('utf-8')).hexdigest()
        if key in self._query_cache:
            return self._query_cache[key].copy()
        emb = self.encoder.encode([q], normalize_embeddings=True, convert_to_numpy=True)
        emb = _normalize(np.asarray(emb, dtype=np.float32))
        self._query_cache[key] = emb
        return emb.copy()

    def retrieve(self, query: str, top_k: int = 10) -> list[dict]:
        if top_k <= 0:
            return []
        q = self.encode_query(query)
        scores = cosine_similarity(q, self.embeddings)[0]
        order = np.argsort(scores)[::-1][:top_k]
        rows = []
        for rank, idx in enumerate(order.tolist(), start=1):
            node_id = self.node_ids[idx]
            rows.append({
                'rank': rank,
                'node_id': node_id,
                'score': float(scores[idx]),
                'text': self.id_to_text.get(node_id, ''),
                'type': self.id_to_type.get(node_id, ''),
            })
        return rows


class LocalBM25Retriever:
    def __init__(self, nodes_df: pd.DataFrame):
        self.nodes_df = nodes_df.reset_index(drop=True).copy()
        self.node_ids = self.nodes_df['node_id'].astype(str).tolist()
        texts = self.nodes_df.get('text', pd.Series([''] * len(self.nodes_df))).fillna('').astype(str).tolist()
        self.id_to_text = dict(zip(self.node_ids, texts))
        self.id_to_type = dict(zip(self.node_ids, self.nodes_df.get('type', pd.Series([''] * len(self.nodes_df))).fillna('').astype(str)))
        self.bm25 = BM25Okapi([_tokenize(t) for t in texts])

    def retrieve(self, query: str, top_k: int = 10) -> list[dict]:
        if top_k <= 0:
            return []
        tokens = _tokenize(str(query).strip())
        scores = np.asarray(self.bm25.get_scores(tokens), dtype=np.float32)
        order = np.argsort(scores)[::-1][:top_k]
        rows = []
        for rank, idx in enumerate(order.tolist(), start=1):
            node_id = self.node_ids[idx]
            rows.append({
                'rank': rank,
                'node_id': node_id,
                'score': float(scores[idx]),
                'text': self.id_to_text.get(node_id, ''),
                'type': self.id_to_type.get(node_id, ''),
            })
        return rows

In [ ]:
# Load config and downstream artifacts
with open(resolve_path('configs/config.yaml'), 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

nodes = pd.read_csv(resolve_path(config['paths']['nodes_csv']))
edges = pd.read_csv(resolve_path(config['paths']['edges_csv']))
node_features = np.load(resolve_path(config['paths']['node_features']))
gnn_embeddings = np.load(resolve_path(config['paths']['final_embeddings']))

embedding_dim = int(config['embedding']['embedding_dim'])
text_embeddings = np.asarray(node_features[:, :embedding_dim], dtype=np.float32)

nodes = nodes.copy()
nodes['node_id'] = nodes['node_id'].astype(str)
edges = edges.copy()
edges['source'] = edges['source'].astype(str)
edges['target'] = edges['target'].astype(str)

node_ids = nodes['node_id'].tolist()
node_to_idx = {node_id: idx for idx, node_id in enumerate(node_ids)}

full_graph = nx.Graph()
full_graph.add_nodes_from(node_ids)
for row in edges.itertuples(index=False):
    if row.source != row.target and row.source in node_to_idx and row.target in node_to_idx:
        full_graph.add_edge(row.source, row.target)

local_only = bool(str(Path.home())) and (
    str(Path.home()) != '' and (
        __import__('os').getenv('HF_HUB_OFFLINE') == '1' or __import__('os').getenv('TRANSFORMERS_OFFLINE') == '1'
    )
)
encoder = SentenceTransformer(config['embedding']['model_name'], local_files_only=local_only)

bm25 = LocalBM25Retriever(nodes)
dense = LocalDenseRetriever(nodes_df=nodes, embeddings=text_embeddings, encoder=encoder)

print(f'nodes={len(nodes)} edges={len(edges)} text_emb={text_embeddings.shape} gnn_emb={gnn_embeddings.shape}')
if not PCST_AVAILABLE:
    print('Warning: pcst_fast not installed in this kernel. Install with: pip install pcst-fast')

In [ ]:
# Shared helpers: query expansion, fusion, gnn expansion, tables, plotting, metrics
LEGAL_SYNONYM_MAP = {
    'ai agent': ['ai system', 'general purpose ai model', 'autonomous agent'],
    'ai agents': ['ai system', 'general purpose ai model', 'autonomous agents'],
    'gpai': ['general purpose ai', 'general purpose ai model'],
    'general-purpose ai': ['general purpose ai', 'gpai'],
    'general-purpose ai model': ['general purpose ai model', 'gpai model'],
    'high-risk': ['high risk', 'high-risk ai system'],
    'high risk': ['high-risk', 'high-risk ai system'],
    'provider': ['provider obligations', 'obligations of providers'],
    'deployer': ['deployer obligations', 'obligations of deployers'],
    'prohibited': ['prohibited ai practices', 'unacceptable risk ai'],
    'ai literacy': ['literacy obligations', 'article 4 ai literacy'],
}


def normalize_legal_query(query: str) -> str:
    text = str(query).strip().lower()
    text = re.sub(r'\bart\.?\b', 'article', text)
    text = re.sub(r'\barts\.?\b', 'articles', text)
    text = re.sub(r'\brec\.?\b', 'recital', text)
    text = re.sub(r'\bch\.?\b', 'chapter', text)
    text = re.sub(r'\bann\.?\b', 'annex', text)
    text = re.sub(r'\bgpai\b', 'general purpose ai', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_citation_expansions(text: str) -> list[str]:
    norm_text = normalize_legal_query(text)
    expansions = set()

    article_pat = re.compile(r'\barticle\s+(\d+[a-z]?(?:\s*\([^)]+\))*)')
    for raw in article_pat.findall(norm_text):
        compact = re.sub(r'\s+', '', raw)
        base = re.match(r'(\d+[a-z]?)', compact)
        if base:
            expansions.add(f"article {base.group(1)}")
        expansions.add(f'article {compact}')

    for raw in re.findall(r'\brecital\s+(\d+[a-z]?)', norm_text):
        expansions.add(f'recital {raw}')
    for raw in re.findall(r'\bchapter\s+([ivxlcdm]+|\d+)\b', norm_text):
        expansions.add(f'chapter {raw}')
    for raw in re.findall(r'\bannex\s+([ivxlcdm]+|\d+)\b', norm_text):
        expansions.add(f'annex {raw}')

    return sorted(expansions)


def expand_legal_query(query: str, max_extra_terms: int = 10) -> str:
    original = str(query).strip()
    if not original:
        return ''
    norm_query = normalize_legal_query(original)
    extras = []
    extras.extend(extract_citation_expansions(norm_query))
    for needle, synonyms in LEGAL_SYNONYM_MAP.items():
        if needle in norm_query:
            extras.extend(synonyms)

    deduped = []
    seen = set()
    for term in extras:
        t = str(term).strip().lower()
        if not t or t in seen:
            continue
        seen.add(t)
        deduped.append(t)
        if len(deduped) >= int(max_extra_terms):
            break

    return original if not deduped else f"{original} {' '.join(deduped)}"


def to_score_map(rows: list[dict]) -> dict[str, float]:
    return {r['node_id']: float(r['score']) for r in rows}


def minmax(score_map: dict[str, float]) -> dict[str, float]:
    if not score_map:
        return {}
    vals = np.array(list(score_map.values()), dtype=float)
    lo, hi = vals.min(), vals.max()
    if hi - lo < 1e-12:
        return {k: 1.0 for k in score_map}
    return {k: float((v - lo) / (hi - lo)) for k, v in score_map.items()}


def align_query_for_gnn(query_emb: np.ndarray) -> np.ndarray:
    q = np.asarray(query_emb, dtype=np.float32).reshape(-1)
    d_q = q.shape[0]
    d_g = int(gnn_embeddings.shape[1])
    if d_q == d_g:
        out = q
    elif d_q > d_g:
        out = q[:d_g]
    else:
        out = np.pad(q, (0, d_g - d_q), mode='constant')
    n = np.linalg.norm(out)
    return out / max(n, 1e-12)


def fuse_text_retrieval(query, top_k=30, bm25_weight=0.4, dense_weight=0.6, query_expansion=True, max_query_expansion_terms=10):
    retrieval_query = expand_legal_query(query, max_extra_terms=max_query_expansion_terms) if query_expansion else query
    bm25_rows = bm25.retrieve(retrieval_query, top_k=top_k)
    dense_rows = dense.retrieve(retrieval_query, top_k=top_k)

    bm25_scores = minmax(to_score_map(bm25_rows))
    dense_scores = minmax(to_score_map(dense_rows))

    merged_ids = sorted(set(bm25_scores) | set(dense_scores))
    fused = []
    for nid in merged_ids:
        score = bm25_weight * bm25_scores.get(nid, 0.0) + dense_weight * dense_scores.get(nid, 0.0)
        fused.append({
            'node_id': nid,
            'score': float(score),
            'bm25_score': float(bm25_scores.get(nid, 0.0)),
            'dense_score': float(dense_scores.get(nid, 0.0)),
        })
    fused.sort(key=lambda x: x['score'], reverse=True)
    return fused, retrieval_query


def gnn_expand_from_seeds(seed_nodes, query_emb, hops=2, max_expand=20, hop_decay=0.85):
    if not seed_nodes:
        return []
    q_gnn = align_query_for_gnn(query_emb)
    seed_set = set(seed_nodes)
    candidate_best = {}

    for seed_rank, seed in enumerate(seed_nodes, start=1):
        if seed not in full_graph or seed not in node_to_idx:
            continue
        seed_weight = 1.0 / float(seed_rank)
        frontier = {seed}
        visited = {seed}

        for hop in range(1, int(hops) + 1):
            nxt = set()
            for src in frontier:
                for nbr in full_graph.neighbors(src):
                    if nbr in visited:
                        continue
                    nxt.add(nbr)
            if not nxt:
                break

            for nbr in nxt:
                if nbr in seed_set or nbr not in node_to_idx:
                    continue
                nbr_idx = node_to_idx[nbr]
                nbr_emb = gnn_embeddings[nbr_idx]
                sim = float(np.dot(q_gnn, nbr_emb) / (np.linalg.norm(nbr_emb) + 1e-12))
                score = sim * (hop_decay ** (hop - 1)) * seed_weight
                current = candidate_best.get(nbr)
                if current is None or score > current['gnn_score']:
                    candidate_best[nbr] = {
                        'node_id': nbr,
                        'gnn_score': float(score),
                        'hop': int(hop),
                        'from_seed': seed,
                    }
            visited.update(nxt)
            frontier = nxt

    expanded = sorted(candidate_best.values(), key=lambda x: x['gnn_score'], reverse=True)
    return expanded[:int(max_expand)]


def expand_k_hops(graph, seeds, k=1):
    seeds = [s for s in seeds if s in graph]
    nodes_in_scope = set(seeds)
    frontier = set(seeds)
    for _ in range(int(k)):
        nxt = set()
        for n in frontier:
            nxt.update(graph.neighbors(n))
        frontier = nxt
        nodes_in_scope.update(nxt)
    return graph.subgraph(nodes_in_scope).copy()


def get_text_rows(node_ids, nodes_df=nodes):
    requested = [str(node_id) for node_id in node_ids]
    order = {node_id: idx for idx, node_id in enumerate(requested)}
    matches = nodes_df[nodes_df['node_id'].isin(requested)].copy()
    matches['request_order'] = matches['node_id'].map(order)
    cols = [c for c in ['node_id', 'title', 'type', 'text'] if c in matches.columns]
    return matches.sort_values('request_order')[cols].reset_index(drop=True)


def plot_subgraph(graph, title, nodes_df=nodes, score_lookup=None):
    type_lookup = nodes_df[['node_id', 'type']].drop_duplicates('node_id').set_index('node_id')['type'].astype(str).to_dict()

    try:
        layout = nx.spring_layout(graph, seed=7)
    except TypeError:
        layout = nx.spring_layout(graph)

    edge_x, edge_y = [], []
    for source, target in graph.edges():
        x0, y0 = layout[source]
        x1, y1 = layout[target]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(width=0.6, color='#888'), hoverinfo='skip')

    palette = {
        'article': '#1f77b4', 'recital': '#ff7f0e', 'chapter': '#2ca02c', 'annex': '#d62728',
        'definition': '#9467bd', 'annex_item': '#8c564b', 'paragraph': '#17becf',
    }
    node_x, node_y, node_text, node_color, node_size = [], [], [], [], []

    for node_id in graph.nodes():
        x, y = layout[node_id]
        node_x.append(x)
        node_y.append(y)

        ntype = type_lookup.get(node_id, 'unknown').lower()
        node_color.append(palette.get(ntype, '#7f7f7f'))

        score = {} if score_lookup is None else score_lookup.get(node_id, {})
        fused = float(score.get('fused_score', 0.0))
        gnn = float(score.get('gnn_score', 0.0))
        prize = float(score.get('prize', 0.0))
        node_size.append(14 + 10 * max(0.0, prize))
        node_text.append(f"{node_id}<br>type={ntype}<br>fused={fused:.4f}<br>gnn={gnn:.4f}<br>prize={prize:.4f}")

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode='markers',
        hoverinfo='text',
        text=node_text,
        marker=dict(color=node_color, size=node_size, line=dict(width=1, color='#333'), opacity=0.9),
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title=title,
        showlegend=False,
        hovermode='closest',
        margin=dict(l=10, r=10, t=50, b=10),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=560,
    )
    return fig


def evaluate_retrieval_from_qrels(predictions, qrels, k_values=(1, 3, 5, 10)):
    k_list = sorted({int(k) for k in k_values if int(k) > 0})
    rows = []

    def _ndcg(ranked, gold, k):
        rel = [1.0 if x in gold else 0.0 for x in ranked[:k]]
        dcg = sum(r / np.log2(i + 2) for i, r in enumerate(rel))
        ideal_len = min(len(gold), k)
        if ideal_len == 0:
            return 0.0
        idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_len))
        return float(dcg / idcg)

    for qid, ranked_ids in predictions.items():
        ranked = [str(x) for x in ranked_ids]
        gold = {str(x) for x in qrels.get(qid, [])}
        row = {'query_id': qid, 'gold_size': len(gold)}

        for k in k_list:
            top = ranked[:k]
            hits = sum(1 for x in top if x in gold)
            row[f'precision@{k}'] = float(hits / k)
            row[f'recall@{k}'] = float(hits / len(gold)) if gold else 0.0
            row[f'ndcg@{k}'] = _ndcg(ranked, gold, k)

        cutoff = k_list[-1]
        mrr = 0.0
        for i, x in enumerate(ranked[:cutoff], start=1):
            if x in gold:
                mrr = 1.0 / float(i)
                break
        row[f'mrr@{cutoff}'] = mrr
        rows.append(row)

    if not rows:
        return {'per_query': pd.DataFrame(), 'macro': {}}

    per_query = pd.DataFrame(rows)
    metric_cols = [c for c in per_query.columns if c not in {'query_id', 'gold_size'}]
    macro = {c: float(per_query[c].mean()) for c in metric_cols}
    macro['queries'] = int(len(per_query))
    return {'per_query': per_query, 'macro': macro}

In [ ]:
# Module 1: text retrieval -> gnn expansion -> unpruned subgraph

def retrieve_unpruned_subgraph(
    query,
    num_retrieved_seeds=5,
    retrieval_top_k=40,
    gnn_expand_hops=2,
    gnn_max_expand=20,
    k_hops=1,
    bm25_weight=0.4,
    dense_weight=0.6,
    query_expansion=True,
    max_query_expansion_terms=10,
):
    fused, retrieval_query = fuse_text_retrieval(
        query,
        top_k=retrieval_top_k,
        bm25_weight=bm25_weight,
        dense_weight=dense_weight,
        query_expansion=query_expansion,
        max_query_expansion_terms=max_query_expansion_terms,
    )

    fused_df = pd.DataFrame(fused)
    seed_nodes = [row['node_id'] for row in fused[:int(num_retrieved_seeds)]]
    query_emb = dense.encode_query(retrieval_query)[0]

    expanded_rows = gnn_expand_from_seeds(
        seed_nodes=seed_nodes,
        query_emb=query_emb,
        hops=gnn_expand_hops,
        max_expand=gnn_max_expand,
        hop_decay=0.85,
    )
    expanded_nodes = [row['node_id'] for row in expanded_rows]

    base_nodes = list(dict.fromkeys(seed_nodes + expanded_nodes + fused_df['node_id'].head(int(retrieval_top_k)).tolist()))
    unpruned_graph = expand_k_hops(full_graph, base_nodes, k=k_hops)

    retrieved_node_ids = list(unpruned_graph.nodes())
    retrieved_edges_df = pd.DataFrame(list(unpruned_graph.edges()), columns=['source', 'target'])

    expanded_df = pd.DataFrame(expanded_rows)
    retrieved_text_df = get_text_rows(retrieved_node_ids)

    if not fused_df.empty:
        retrieved_text_df = retrieved_text_df.merge(
            fused_df[['node_id', 'score', 'bm25_score', 'dense_score']],
            on='node_id',
            how='left',
        )
    if not expanded_df.empty:
        retrieved_text_df = retrieved_text_df.merge(
            expanded_df[['node_id', 'gnn_score', 'hop', 'from_seed']],
            on='node_id',
            how='left',
        )

    retrieved_text_df['is_seed'] = retrieved_text_df['node_id'].isin(seed_nodes)
    retrieved_text_df['is_expanded'] = retrieved_text_df['node_id'].isin(expanded_nodes)
    retrieved_text_df['score'] = retrieved_text_df['score'].fillna(0.0)
    retrieved_text_df['gnn_score'] = retrieved_text_df['gnn_score'].fillna(0.0)

    retrieved_text_df = retrieved_text_df.sort_values(
        ['score', 'gnn_score', 'is_seed', 'is_expanded'],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    score_lookup = (
        retrieved_text_df[['node_id', 'score', 'gnn_score']]
        .drop_duplicates('node_id')
        .set_index('node_id')
        .rename(columns={'score': 'fused_score'})
        .to_dict('index')
    )
    retrieved_figure = plot_subgraph(unpruned_graph, title=f'Unpruned retrieved subgraph for query: {query}', score_lookup=score_lookup)

    return {
        'query': query,
        'retrieval_query': retrieval_query,
        'query_emb': query_emb,
        'seed_nodes': seed_nodes,
        'expanded_nodes': expanded_nodes,
        'fused_seed_scores': fused_df.head(max(int(num_retrieved_seeds), 15)),
        'gnn_expansion_scores': expanded_df,
        'subgraph': unpruned_graph,
        'subgraph_node_ids': retrieved_node_ids,
        'subgraph_edges_df': retrieved_edges_df,
        'text_df': retrieved_text_df,
        'figure': retrieved_figure,
    }

In [ ]:
# Module 2: PCST prune on retrieved subgraph

def run_node_prize_pcst(graph, prizes, edge_cost=1.0, pruning='strong'):
    if not PCST_AVAILABLE:
        raise ImportError('pcst_fast is not installed. Install with: pip install pcst-fast')

    node_list = list(graph.nodes())
    node_to_idx = {node_id: idx for idx, node_id in enumerate(node_list)}
    edge_list = list(graph.edges())

    if not edge_list:
        return node_list, []

    edge_array = np.asarray([(node_to_idx[s], node_to_idx[t]) for s, t in edge_list], dtype=np.int32)
    prize_array = np.asarray(prizes, dtype=np.float64)
    edge_costs = np.full(len(edge_list), float(edge_cost), dtype=np.float64)

    selected_vertex_idx, selected_edge_idx = pcst_fast(edge_array, prize_array, edge_costs, -1, 1, pruning, 0)

    selected_nodes = [node_list[idx] for idx in selected_vertex_idx]
    selected_node_set = set(selected_nodes)
    selected_edges = []
    for edge_idx in selected_edge_idx:
        source, target = edge_list[edge_idx]
        if source in selected_node_set and target in selected_node_set:
            selected_edges.append((source, target))

    return selected_nodes, selected_edges


def prune_retrieved_subgraph(
    retrieval_result,
    prize_top_k=None,
    edge_cost=1.0,
    fused_weight=0.70,
    gnn_weight=0.20,
    sim_weight=0.10,
):
    candidate_graph = retrieval_result['subgraph']
    candidate_node_ids = retrieval_result['subgraph_node_ids']

    text_df = retrieval_result['text_df'][['node_id', 'score', 'gnn_score']].copy()
    text_df = text_df.drop_duplicates('node_id')

    fused_map = text_df.set_index('node_id')['score'].to_dict()
    gnn_map = text_df.set_index('node_id')['gnn_score'].to_dict()

    fused_vals = np.array([float(fused_map.get(node_id, 0.0)) for node_id in candidate_node_ids], dtype=float)
    gnn_vals = np.array([float(gnn_map.get(node_id, 0.0)) for node_id in candidate_node_ids], dtype=float)

    f_norm = fused_vals if np.ptp(fused_vals) < 1e-12 else (fused_vals - fused_vals.min()) / (fused_vals.max() - fused_vals.min())
    g_norm = gnn_vals if np.ptp(gnn_vals) < 1e-12 else (gnn_vals - gnn_vals.min()) / (gnn_vals.max() - gnn_vals.min())

    q_text = np.asarray(retrieval_result['query_emb'], dtype=np.float32).reshape(-1)
    node_text_mat = np.vstack([text_embeddings[node_to_idx[node_id]] for node_id in candidate_node_ids])
    sim_vals = cosine_similarity([q_text], node_text_mat)[0]
    s_norm = (sim_vals + 1.0) / 2.0

    prizes = fused_weight * f_norm + gnn_weight * g_norm + sim_weight * s_norm

    if prize_top_k is not None and int(prize_top_k) > 0 and len(prizes) > int(prize_top_k):
        keep_idx = np.argsort(prizes)[::-1][:int(prize_top_k)]
        masked = np.zeros_like(prizes)
        masked[keep_idx] = prizes[keep_idx]
        prizes = masked

    selected_nodes, selected_edges = run_node_prize_pcst(candidate_graph, prizes, edge_cost=edge_cost, pruning='strong')

    node_scores_df = pd.DataFrame({
        'node_id': candidate_node_ids,
        'fused_score': f_norm,
        'gnn_score': g_norm,
        'similarity': s_norm,
        'prize': prizes,
    }).sort_values(['prize', 'fused_score', 'gnn_score', 'similarity'], ascending=False)

    selected_edges_df = pd.DataFrame(selected_edges, columns=['source', 'target'])
    pruned_graph = nx.Graph()
    pruned_graph.add_nodes_from(selected_nodes)
    pruned_graph.add_edges_from(selected_edges)

    pruned_text_df = get_text_rows(selected_nodes)
    pruned_text_df = pruned_text_df.merge(
        node_scores_df[['node_id', 'fused_score', 'gnn_score', 'similarity', 'prize']],
        on='node_id',
        how='left',
    ).sort_values(['prize', 'fused_score', 'gnn_score', 'similarity'], ascending=False).reset_index(drop=True)

    score_lookup = node_scores_df.set_index('node_id').to_dict('index')
    pruned_figure = plot_subgraph(pruned_graph, title=f"PCST-pruned subgraph for query: {retrieval_result['query']}", score_lookup=score_lookup)

    return {
        'query': retrieval_result['query'],
        'selected_nodes': selected_nodes,
        'selected_edges_df': selected_edges_df,
        'node_scores_df': node_scores_df,
        'pruned_graph': pruned_graph,
        'text_df': pruned_text_df,
        'figure': pruned_figure,
    }

In [ ]:
# Example (same interaction pattern as 03_retrieval_prune.ipynb)
retrieval_result = retrieve_unpruned_subgraph(
    query='What differentiates AI systems and AI models?',
    num_retrieved_seeds=5,
    retrieval_top_k=40,
    gnn_expand_hops=2,
    gnn_max_expand=20,
    k_hops=1,
    bm25_weight=0.4,
    dense_weight=0.6,
)

retrieval_result['text_df'].head(30)

In [ ]:
retrieval_result['figure']


In [ ]:
pruned_result = prune_retrieved_subgraph(
    retrieval_result,
    prize_top_k=None,
    edge_cost=0.8,
    fused_weight=0.70,
    gnn_weight=0.20,
    sim_weight=0.10,
)

pruned_result['text_df'].head(30)

In [ ]:
pruned_result['figure']


## Notes

- This notebook is now self-contained: no `src/retrieval/retriever.py` import.
- Pipeline is exactly: `query -> dense/BM25 text retrieval -> GNN expansion -> PCST`.
- `text_df` and `figure` outputs are kept in the same style as `03_retrieval_prune.ipynb`.

## 16-Query Benchmark

Runs your 16-query test set and reports macro + per-query retrieval metrics for:
- unpruned subgraph ranking
- PCST-pruned ranking

In [ ]:
# 16-query test set from the table provided in chat
TEST_SET_16 = [
    {
        'query_id': 'q1',
        'query': 'How are AI Agents addressed within the AI Act',
        'recitals': 'Recital 97',
        'articles': 'Article 3(1); Article 3(63); Article 5(1)(a),(b); Article 50; Article 51(1)(b)',
        'chapters': 'Chapter III',
        'annexes': 'Annex XIII (point e)',
    },
    {
        'query_id': 'q2',
        'query': 'Are logic-based or knowledge-based approaches also covered by the AI System definition?',
        'recitals': 'Recital 12',
        'articles': '',
        'chapters': '',
        'annexes': '',
    },
    {
        'query_id': 'q3',
        'query': 'What distinguishes an AI model from an AI System? Will guidance be issued?',
        'recitals': 'Recital 12; Recital 97; Recital 99',
        'articles': 'Article 3(1); Article 3(63)',
        'chapters': '',
        'annexes': '',
    },
    {
        'query_id': 'q4',
        'query': 'What is AI literacy as described in Article 4 of the AI Act?',
        'recitals': '',
        'articles': 'Article 4; Article 3(56)',
        'chapters': '',
        'annexes': '',
    },
    {
        'query_id': 'q5',
        'query': 'What AI systems are prohibited?',
        'recitals': '',
        'articles': 'Article 5',
        'chapters': '',
        'annexes': '',
    },
    {
        'query_id': 'q6',
        'query': 'What documentation/assessments are needed for providers of high-risk AI systems to comply with the AI Act?',
        'recitals': '',
        'articles': 'Article 16; Articles 8-15; Articles 17-20',
        'chapters': 'Chapter III',
        'annexes': '',
    },
    {
        'query_id': 'q7',
        'query': 'Does location data quality as biometric data under the AI Act?',
        'recitals': '',
        'articles': 'Article 5(1)(g)',
        'chapters': '',
        'annexes': 'Annex III (point 1)',
    },
    {
        'query_id': 'q8',
        'query': 'Which AI models does the AI Act apply to?',
        'recitals': '',
        'articles': 'Article 2(6); Article 3(9); Article 3(10)',
        'chapters': '',
        'annexes': '',
    },
    {
        'query_id': 'q9',
        'query': 'When does a model qualify as a general-purpose AI model?',
        'recitals': '',
        'articles': 'Article 3(63)',
        'chapters': '',
        'annexes': '',
    },
    {
        'query_id': 'q10',
        'query': 'When is a general-purpose AI model one with systemic risk?',
        'recitals': '',
        'articles': 'Article 51(2); Article 51(1)(b)',
        'chapters': '',
        'annexes': '',
    },
    {
        'query_id': 'q11',
        'query': 'What are the obligations of the provider if a serious incident involving its model occurs?',
        'recitals': '',
        'articles': 'Article 55(1)(c),(b),(d); Article 3(49)(a-d)',
        'chapters': 'Chapter V',
        'annexes': '',
    },
    {
        'query_id': 'q12',
        'query': 'What does the AI Act regulate in terms of AI literacy and governance, and what is the impact on businesses?',
        'recitals': '',
        'articles': 'Article 4; Article 10',
        'chapters': '',
        'annexes': '',
    },
    {
        'query_id': 'q13',
        'query': 'What are the specific requirements for general-purpose AI models (GPAI) under the AI Act?',
        'recitals': '',
        'articles': 'Article 3(3); Article 3(9); Article 53',
        'chapters': '',
        'annexes': '',
    },
    {
        'query_id': 'q14',
        'query': 'What are the specific requirements for the use of AI systems in critical infrastructure?',
        'recitals': '',
        'articles': 'Article 6; Article 49(2)',
        'chapters': '',
        'annexes': '',
    },
    {
        'query_id': 'q15',
        'query': 'What are the obligations for non-EU providers of AI systems when placing their products on the EU market?',
        'recitals': '',
        'articles': 'Article 54',
        'chapters': '',
        'annexes': '',
    },
    {
        'query_id': 'q16',
        'query': 'What are the specific obligations for providers and deployers in cases where AI systems are customized for individual clients?',
        'recitals': '',
        'articles': 'Article 25; Article 6; Article 25(1)(a); Article 25(1)(b); Article 25(1)(c)',
        'chapters': '',
        'annexes': '',
    },
]


def build_gold_from_table(recitals='', articles='', chapters='', annexes=''):
    text = '; '.join([str(recitals), str(articles), str(chapters), str(annexes)])
    text = text.replace('–', '-').replace('—', ' ')
    gold = set()

    chapter_articles = {
        ch: set(nodes[(nodes['type'].astype(str).str.lower() == 'article') & (nodes['chapter'].astype(str) == ch)]['node_id'].astype(str))
        for ch in nodes['chapter'].dropna().astype(str).unique()
    }

    for m in re.finditer(r'Recital\s+(\d+)', text, flags=re.I):
        nid = f"recital_{m.group(1)}"
        if nid in node_to_idx:
            gold.add(nid)

    for m in re.finditer(r'Annex\s+([IVXLCDM]+|\d+)', text, flags=re.I):
        nid = f"annex_{m.group(1).upper()}"
        if nid in node_to_idx:
            gold.add(nid)

    for m in re.finditer(r'Chapter\s+([IVXLCDM]+|\d+)', text, flags=re.I):
        gold.update(chapter_articles.get(m.group(1).upper(), set()))

    for m in re.finditer(r'Articles?\s+(\d+)\s*-\s*(\d+)', text, flags=re.I):
        a, b = sorted([int(m.group(1)), int(m.group(2))])
        for x in range(a, b + 1):
            nid = f'article_{x}'
            if nid in node_to_idx:
                gold.add(nid)

    for m in re.finditer(r'Article\s+(\d+)([^;]*)', text, flags=re.I):
        art = m.group(1)
        tail = m.group(2)
        nid = f'article_{art}'
        if nid in node_to_idx:
            gold.add(nid)
        para_match = re.search(r'\((\d+[a-z]?)\)', tail, flags=re.I)
        if para_match:
            para_nid = f"article_{art}_para_{para_match.group(1)}"
            if para_nid in node_to_idx:
                gold.add(para_nid)

    return sorted(gold)


PRED_UNPRUNED = {}
PRED_PRUNED = {}
QRELS = {}
ROWS = []

for row in TEST_SET_16:
    qid = row['query_id']
    query = row['query']

    gold = build_gold_from_table(
        recitals=row.get('recitals', ''),
        articles=row.get('articles', ''),
        chapters=row.get('chapters', ''),
        annexes=row.get('annexes', ''),
    )
    QRELS[qid] = gold

    unpruned = retrieve_unpruned_subgraph(
        query=query,
        num_retrieved_seeds=5,
        retrieval_top_k=40,
        gnn_expand_hops=2,
        gnn_max_expand=20,
        k_hops=1,
        bm25_weight=0.4,
        dense_weight=0.6,
        query_expansion=True,
    )
    pruned = prune_retrieved_subgraph(
        unpruned,
        prize_top_k=None,
        edge_cost=0.8,
        fused_weight=0.70,
        gnn_weight=0.20,
        sim_weight=0.10,
    )

    un_nodes = unpruned['text_df']['node_id'].astype(str).tolist()[:10]
    pr_nodes = pruned['text_df']['node_id'].astype(str).tolist()[:10]

    PRED_UNPRUNED[qid] = un_nodes
    PRED_PRUNED[qid] = pr_nodes

    ROWS.append({
        'query_id': qid,
        'query': query,
        'gold_size': len(gold),
        'unpruned_top10': un_nodes,
        'pruned_top10': pr_nodes,
    })

metrics_un = evaluate_retrieval_from_qrels(PRED_UNPRUNED, QRELS, k_values=(5, 10))
metrics_pr = evaluate_retrieval_from_qrels(PRED_PRUNED, QRELS, k_values=(5, 10))

macro_df = pd.DataFrame([
    {'pipeline': 'unpruned', **metrics_un['macro']},
    {'pipeline': 'pruned', **metrics_pr['macro']},
]).set_index('pipeline')

per_query_df = metrics_pr['per_query'][['query_id', 'gold_size', 'recall@5', 'recall@10', 'precision@5', 'ndcg@10', 'mrr@10']].copy()

try:
    display(macro_df)
    display(per_query_df.sort_values('query_id').reset_index(drop=True))
except NameError:
    print(macro_df)
    print(per_query_df.sort_values('query_id').reset_index(drop=True).to_string(index=False))